In [1]:
import math
import numpy as np
import pandas as pd

print("Libraries imported successfully! ✅")


Libraries imported successfully! ✅


## 2. Load multi-city recommendation data

In [2]:
df = pd.read_csv(
    "../data/processed/multi_city_recommendation_features.csv"
)

print("Dataset shape:", df.shape)
print("\nCities:")
print(df["city"].value_counts())


Dataset shape: (120, 27)

Cities:
city
Manali       20
Goa          20
Jaipur       20
Udaipur      20
Rishikesh    20
Shimla       20
Name: count, dtype: int64


## 3. Define configurable itinerary settings

In [3]:
AVERAGE_SPEED_KMPH = 25
DAY_START_MINUTES = 9 * 60
MAX_DAY_MINUTES = 7 * 60

print("Average speed:", AVERAGE_SPEED_KMPH, "km/h")
print("Daily planning budget:", MAX_DAY_MINUTES, "minutes")


Average speed: 25 km/h
Daily planning budget: 420 minutes


## 4. Haversine distance

In [4]:
def haversine_km(lat1, lon1, lat2, lon2):
    earth_radius_km = 6371.0

    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(
        math.sqrt(a),
        math.sqrt(1 - a)
    )

    return earth_radius_km * c


## 5. Time formatter

In [5]:
def format_time(minutes):
    minutes = int(round(minutes))

    hour = (minutes // 60) % 24
    minute = minutes % 60

    suffix = "AM" if hour < 12 else "PM"
    display_hour = hour % 12 or 12

    return f"{display_hour}:{minute:02d} {suffix}"


## 6. Select candidates for one city

The recommender already calculated `final_score`. We use that score to create the candidate pool.

We keep more candidates than the final number of stops so the optimizer has room to spread attractions across days.


In [6]:
def select_city_candidates(
    city,
    candidate_count=15
):
    city_clean = city.strip().lower()

    city_df = df[
        df["city"].str.lower() == city_clean
    ].copy()

    if city_df.empty:
        raise ValueError(
            f"City '{city}' was not found. "
            f"Available cities: "
            f"{sorted(df['city'].unique())}"
        )

    return (
        city_df
        .sort_values(
            "final_score",
            ascending=False
        )
        .head(
            min(candidate_count, len(city_df))
        )
        .reset_index(drop=True)
    )


## 7. Build the travel-time matrix for a city

In [7]:
def build_travel_time_matrix(city_df):
    n = len(city_df)

    matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            distance = haversine_km(
                city_df.loc[i, "latitude"],
                city_df.loc[i, "longitude"],
                city_df.loc[j, "latitude"],
                city_df.loc[j, "longitude"]
            )

            matrix[i, j] = (
                distance
                / AVERAGE_SPEED_KMPH
                * 60
            )

    return matrix


## 8. Generate one city's multi-day itinerary

The planner:

1. starts each day with a strong remaining candidate
2. selects nearby high-scoring places
3. respects the daily time budget
4. removes scheduled places from later days

This is a **greedy heuristic**, not a guaranteed optimal solution.


In [8]:
def generate_city_itinerary(
    city,
    days=3,
    candidate_count=12,
    max_day_minutes=MAX_DAY_MINUTES
):
    candidates = select_city_candidates(
        city,
        candidate_count=candidate_count
    )

    travel_matrix = build_travel_time_matrix(
        candidates
    )

    remaining = set(
        range(len(candidates))
    )

    itinerary_rows = []

    for day in range(1, days + 1):

        if not remaining:
            break

        # Start the day from a strong remaining candidate.
        anchor = max(
            remaining,
            key=lambda idx:
            candidates.loc[idx, "final_score"]
        )

        current = anchor
        used_minutes = 0

        while remaining:

            feasible = []

            for idx in remaining:

                travel = (
                    0
                    if used_minutes == 0
                    else travel_matrix[current, idx]
                )

                visit = float(
                    candidates.loc[
                        idx,
                        "estimated_visit_minutes"
                    ]
                )

                total = travel + visit

                if (
                    used_minutes + total
                    <= max_day_minutes
                ):
                    efficiency = (
                        candidates.loc[
                            idx,
                            "final_score"
                        ]
                        / (1 + travel)
                    )

                    feasible.append(
                        (
                            idx,
                            travel,
                            visit,
                            efficiency
                        )
                    )

            if not feasible:
                break

            idx, travel, visit, _ = max(
                feasible,
                key=lambda item: item[3]
            )

            arrival = (
                DAY_START_MINUTES
                + used_minutes
                + travel
            )

            departure = arrival + visit

            itinerary_rows.append({
                "city": city,
                "day": day,
                "stop": sum(
                    1
                    for row in itinerary_rows
                    if row["day"] == day
                    and row["city"] == city
                ) + 1,
                "place": candidates.loc[
                    idx, "name"
                ],
                "activity_type": candidates.loc[
                    idx, "activity_type"
                ],
                "arrival": format_time(
                    arrival
                ),
                "departure": format_time(
                    departure
                ),
                "travel_before_minutes":
                    round(travel, 1),
                "visit_minutes":
                    int(visit),
                "estimated_price_level":
                    int(
                        candidates.loc[
                            idx,
                            "estimated_price_level"
                        ]
                    ),
                "final_score":
                    round(
                        float(
                            candidates.loc[
                                idx,
                                "final_score"
                            ]
                        ),
                        4
                    ),
                "latitude":
                    float(
                        candidates.loc[
                            idx,
                            "latitude"
                        ]
                    ),
                "longitude":
                    float(
                        candidates.loc[
                            idx,
                            "longitude"
                        ]
                    )
            })

            used_minutes += total
            current = idx
            remaining.remove(idx)

    return pd.DataFrame(
        itinerary_rows
    )


## 9. Test Goa 🌴

In [9]:
goa_itinerary = generate_city_itinerary(
    "Goa",
    days=3,
    candidate_count=12
)

goa_itinerary


KeyError: 'final_score'

## 10. Verify Goa contains only Goa places

In [ ]:
print(
    "Cities returned:",
    goa_itinerary["city"].unique()
)

assert (
    goa_itinerary["city"]
    .eq("Goa")
    .all()
)

print("✅ Goa city filter verified")


## 11. Test Jaipur 🏰

In [ ]:
jaipur_itinerary = generate_city_itinerary(
    "Jaipur",
    days=3,
    candidate_count=12
)

jaipur_itinerary


## 12. Test all supported cities

In [ ]:
supported_cities = sorted(
    df["city"].unique()
)

city_itineraries = {}

for city in supported_cities:

    plan = generate_city_itinerary(
        city,
        days=3,
        candidate_count=12
    )

    city_itineraries[city] = plan

    print(
        f"{city:12} → "
        f"{len(plan)} scheduled stops"
    )

print("\n✅ All city itineraries generated")


## 13. Check day distribution for every city

In [ ]:
distribution_rows = []

for city, plan in city_itineraries.items():

    if plan.empty:
        continue

    counts = (
        plan
        .groupby("day")
        .size()
        .to_dict()
    )

    for day in range(1, 4):
        distribution_rows.append({
            "city": city,
            "day": day,
            "stops": counts.get(day, 0)
        })

distribution_df = pd.DataFrame(
    distribution_rows
)

distribution_df


## 14. Check daily time budgets

In [ ]:
all_plans = pd.concat(
    city_itineraries.values(),
    ignore_index=True
)

daily_time = (
    all_plans
    .assign(
        total_minutes=lambda x:
            x["travel_before_minutes"]
            + x["visit_minutes"]
    )
    .groupby(["city", "day"])
    ["total_minutes"]
    .sum()
    .reset_index()
)

daily_time["within_budget"] = (
    daily_time["total_minutes"]
    <= MAX_DAY_MINUTES
)

daily_time


## 15. Check that no city is mixed

This is an important data-integrity test.

Every itinerary must contain exactly one city.


In [ ]:
city_integrity = (
    all_plans
    .groupby("city")["city"]
    .nunique()
    .reset_index(name="unique_city_values")
)

city_integrity["valid"] = (
    city_integrity["unique_city_values"] == 1
)

city_integrity


## 16. Create itinerary statistics

In [ ]:
itinerary_statistics = (
    all_plans
    .groupby("city")
    .agg(
        scheduled_places=("place", "count"),
        total_visit_minutes=("visit_minutes", "sum"),
        estimated_travel_minutes=(
            "travel_before_minutes",
            "sum"
        ),
        average_score=("final_score", "mean"),
        total_price_units=(
            "estimated_price_level",
            "sum"
        )
    )
    .reset_index()
)

itinerary_statistics


## 17. Save city-aware itineraries

Two outputs are created:

1. one unified CSV
2. one CSV per city



In [ ]:
unified_path = (
    "../data/processed/"
    "multi_city_itineraries.csv"
)

all_plans.to_csv(
    unified_path,
    index=False
)

city_dir = Path(
    "../data/processed/city_itineraries"
)

city_dir.mkdir(
    parents=True,
    exist_ok=True
)

for city, plan in city_itineraries.items():

    filename = (
        city.lower()
        .replace(" ", "_")
        + "_itinerary.csv"
    )

    plan.to_csv(
        city_dir / filename,
        index=False
    )

print(
    f"✅ Unified itineraries saved: {unified_path}"
)

print(
    f"✅ Per-city itineraries saved: {city_dir}"
)


# 🎯 Milestone

TravelMate is now city-aware across both recommendation and itinerary planning:

```text
User destination
      ↓
City filter
      ↓
Destination-specific recommendations
      ↓
Destination-specific itinerary
      ↓
📅 Day-by-day plan
```

### Next engineering step

We can now update the **FastAPI backend** so `/recommend` and `/itinerary` accept any supported city and use the correct city-specific candidates.

Then the Streamlit frontend can safely provide a city selector instead of the temporary Manali-only restriction.

After that, we should improve the data quality and add better route/start-point handling before deployment.
